In [11]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Define relative paths based on the notebook's location
DATA_DIR = "../datasets/"
MODEL_DIR = "../../backend/models/"

In [12]:
print("Loading Enron dataset...")
enron_path = os.path.join(DATA_DIR, "Enron.csv") # Ensure exact casing matches your file
df_enron = pd.read_csv(enron_path)

# Handle missing values (e.g., emails with no subject)
df_enron['subject'] = df_enron['subject'].fillna('')
df_enron['body'] = df_enron['body'].fillna('')

# Combine subject and body for maximum NLP context
df_enron['content'] = df_enron['subject'] + " " + df_enron['body']

# Keep only what we need and ensure the label is numeric
df_enron = df_enron[['content', 'label']]
df_enron['label'] = pd.to_numeric(df_enron['label'], errors='coerce')

print(f"Enron data loaded: {len(df_enron)} rows.")

Loading Enron dataset...
Enron data loaded: 29767 rows.


In [13]:
print("Loading SMS dataset...")
sms_path = os.path.join(DATA_DIR, "spamHamSMS.csv")

# Standard Kaggle SMS dataset uses 'v1' and 'v2' and 'latin-1' encoding
df_sms = pd.read_csv(sms_path, encoding='latin-1')
df_sms = df_sms[['v1', 'v2']]

# Rename columns to match Enron
df_sms.columns = ['label_text', 'content']

# Map 'spam' to 1 (Phishing) and 'ham' to 0 (Safe)
df_sms['label'] = df_sms['label_text'].map({'spam': 1, 'ham': 0})
df_sms = df_sms[['content', 'label']]

print(f"SMS data loaded: {len(df_sms)} rows.")

Loading SMS dataset...
SMS data loaded: 5572 rows.


In [14]:
# Merge both datasets into one massive training set
df_combined = pd.concat([df_enron, df_sms], ignore_index=True)

# Drop any rows that failed conversion and became NaN
df_combined.dropna(subset=['content', 'label'], inplace=True)
df_combined['label'] = df_combined['label'].astype(int)

print(f"Total Combined Dataset: {len(df_combined)} rows.")

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    df_combined['content'], 
    df_combined['label'], 
    test_size=0.2, 
    random_state=42
)
print("Data successfully split for training and testing.")

Total Combined Dataset: 35339 rows.
Data successfully split for training and testing.


In [15]:
print("Step 1: Vectorizing text using TF-IDF...")
# We use max_features=5000 to keep the model lightweight for the backend
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit the vectorizer on training data, transform both train and test
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Step 2: Training the Logistic Regression model...")
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate the model
predictions = model.predict(X_test_vec)
acc = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {acc * 100:.2f}%")

Step 1: Vectorizing text using TF-IDF...
Step 2: Training the Logistic Regression model...
Model Accuracy: 97.11%


In [16]:
print("Exporting models for the backend API...")

# Ensure the target directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

vectorizer_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
model_path = os.path.join(MODEL_DIR, "phishing_model.pkl")

# Save the files
joblib.dump(vectorizer, vectorizer_path)
joblib.dump(model, model_path)

print(f"✅ SUCCESS! Files saved to:")
print(f" - {os.path.abspath(vectorizer_path)}")
print(f" - {os.path.abspath(model_path)}")

Exporting models for the backend API...
✅ SUCCESS! Files saved to:
 - d:\Projects\IDTHP\CyberSentinel-ai\backend\models\tfidf_vectorizer.pkl
 - d:\Projects\IDTHP\CyberSentinel-ai\backend\models\phishing_model.pkl
